# Workflows
While submitting individual Python functions to the job scheduler is nice, one of the most under used features of modern jobs schedulers is the management of dependencies. You can read more about the dependency management in SLURM in their [documentation](https://slurm.schedmd.com/sbatch.html#OPT_dependency). Our hypothesis is that while people understand that their tasks depend on each other, the usage of the dependency feature or at least the automated usage of this feature in shell scripts is too complex so people instead manage the dependencies manually. To address this challenge executorlib implements the management of dependencies based on using the output of one function as an input to another function.

## Dependencies 

In [1]:
def calc_add(a, b):
    return a + b

In [2]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor() as exe:
    future = 0
    for i in range(1, 4):
        future = exe.submit(calc_add, i, future)
    print(future.result())

6


In [3]:
with SingleNodeExecutor(plot_dependency_graph=True) as exe:
    future = 0
    for i in range(1, 4):
        future = exe.submit(calc_add, i, future)
    print(future.result())

None


ModuleNotFoundError: No module named 'networkx'

## FluxClusterExecutor
The `FluxClusterExecutor` and the `SlurmClusterExecutor` communicate their dependencies directly to the job scheduler. So even when the Python process is stopped the future objects are still executed. Then after the execution the Python process can be restarted and the corresponding future objects are reloaded.  

In [4]:
from executorlib import FluxClusterExecutor

with FluxClusterExecutor() as exe:
    future = 0
    for i in range(1, 4):
        future = exe.submit(calc_add, i, future)
    print(future.result())

6


In [5]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ6VU2j3D jovyan   executorl+  R      1      1   0.949s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6SKgFNo jovyan   executorl+ CD      1      1   1.186s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6P2Rr2K jovyan   executorl+ CD      1      1   1.192s jupyter-janj-lanl-executorlib-tutorial-5rv11ome


To demonstrate this we are going to use the `FluxClusterExecutor` without the `with` statement, but first we have to clear the cache to remove the already existing calculation:

In [6]:
import shutil
shutil.rmtree("executorlib_cache")

In [7]:
exe = FluxClusterExecutor()
future = 0
for i in range(1, 4):
    future = exe.submit(calc_add, i, future)
exe.shutdown(wait=False, cancel_futures=False)

In [8]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒB7Yxrdm jovyan   executorl+  D             -        - depends:after-success=382956732416
    ƒB4TaMY3 jovyan   executorl+  R      1      1   0.852s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB15szLX jovyan   executorl+ CD      1      1   1.220s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6VU2j3D jovyan   executorl+ CD      1      1   1.170s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6SKgFNo jovyan   executorl+ CD      1      1   1.186s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6P2Rr2K jovyan   executorl+ CD      1      1   1.192s jupyter-janj-lanl-executorlib-tutorial-5rv11ome


In [9]:
exe = FluxClusterExecutor()
future = 0
for i in range(1, 4):
    future = exe.submit(calc_add, i, future)
print(future.result())

6


In [10]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒB7Yxrdm jovyan   executorl+ CD      1      1   1.193s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB4TaMY3 jovyan   executorl+ CD      1      1   1.182s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB15szLX jovyan   executorl+ CD      1      1   1.220s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6VU2j3D jovyan   executorl+ CD      1      1   1.170s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6SKgFNo jovyan   executorl+ CD      1      1   1.186s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6P2Rr2K jovyan   executorl+ CD      1      1   1.192s jupyter-janj-lanl-executorlib-tutorial-5rv11ome


## FluxJobExecutor

In [11]:
from executorlib import FluxJobExecutor

with FluxJobExecutor() as exe:
    future = 0
    for i in range(1, 4):
        future = exe.submit(calc_add, i, future)
    print(future.result())

6


In [12]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒKiqWzBH jovyan   python     CD      1      1   0.154s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒKefs1F5 jovyan   python     CD      1      1   0.153s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒKaJM847 jovyan   python     CD      1      1   0.157s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB7Yxrdm jovyan   executorl+ CD      1      1   1.193s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB4TaMY3 jovyan   executorl+ CD      1      1   1.182s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒB15szLX jovyan   executorl+ CD      1      1   1.220s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6VU2j3D jovyan   executorl+ CD      1      1   1.170s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6SKgFNo jovyan   executorl+ CD      1      1   1.186s jupyter-janj-lanl-executorlib-tutorial-5rv11ome
    ƒ6P2Rr2K jovyan   executorl+ CD      1      1   1.192s jupyter-janj-lanl-exe

## Split future
In many cases it is not directly possible to use the output of one function as an input of another function. For example a function might have multiple outputs, a list as an output where you only want to use one element of the list as an input to the next function or a dictionary as an output where you only want to use one value of the dictionary as an input of the next function. To address this challenge executorlib introduces two special functions `split_future()` and `get_item_from_future()`. They are demonstrated below with the `SingleNodeExecutor` but work equally well for all different types of executors.

In [13]:
from executorlib import split_future, get_item_from_future

In [14]:
def function_with_multiple_outputs(i):
    return "a", "b", i

In [15]:
with SingleNodeExecutor() as exe:
    future = exe.submit(function_with_multiple_outputs, 15)
    f1, f2, f3 = split_future(future=future, n=3)
    print(f1.result(), f2.result(), f3.result())

a b 15


In [16]:
def function_with_a_dict_as_output(i):
    return {"a": 1, "b": 2, "c": i}

In [17]:
with SingleNodeExecutor() as exe:
    future_dict = exe.submit(function_with_a_dict_as_output, 15)
    f1 = get_item_from_future(future=future_dict, key="a")
    f2 = get_item_from_future(future=future_dict, key="b")
    f3 = get_item_from_future(future=future_dict, key="c")
    print(f1.result(), f2.result(), f3.result())

1 2 15


## Exascale
A typical task for exascale computing is submitting a large number of tasks and then once a subset of these tasks are completed already start processing those while the remaining tasks are still waiting in the queue or are currently running. 

In [19]:
from time import sleep

In [20]:
def reply(i):
    if i % 5 == 0:
        sleep(5)
    return i

In [23]:
def sum_and_elements(lst):
    return sum(lst), lst

In [24]:
with FluxJobExecutor(max_workers=2) as exe:
    future_individual_lst = [
        exe.submit(reply, i) for i in range(10)
    ]
    future_group_lst = [
        exe.submit(sum_and_elements, f) for f in exe.batched(future_individual_lst, n=3)
    ]
    print([f.result() for f in future_group_lst])

[(6, [1, 2, 3]), (10, [0, 4, 6]), (20, [5, 7, 8]), (9, [9])]
